In [231]:
import glob
import pandas as pd
from functools import reduce
import os
import geopandas as gpd

pasta_arabica = r'C:/Users/gabriel_lc/Desktop/saffraz-k-means/dados-brutos/arabica'
pasta_canephora = r'C:/Users/gabriel_lc/Desktop/saffraz-k-means/dados-brutos/canephora'

arquivos_arabica = glob.glob(pasta_arabica + '/*.xlsx')
arquivos_canephora = glob.glob(pasta_canephora + '/*.xlsx')

In [232]:
def trata_excel(caminho):
    
    df = pd.read_excel(caminho, header=None)
    
    linha1 = df.iloc[1]  # variável
    linha3 = df.iloc[3]  # ano
    linha4 = df.iloc[4]  # tipo
    
    linha1 = linha1.ffill()
    linha3 = linha3.ffill()
    linha4 = linha4.ffill()
    
    novas_colunas = [
        f"{nome}_{ano}_{tipo}" if pd.notna(ano) else nome
        for nome, ano, tipo in zip(linha1, linha3, linha4)
    ]
    
    df.columns = novas_colunas
    
    df = df.iloc[5:].reset_index(drop=True)
    
    df.rename(columns={df.columns[0]: 'mesorregiao'}, inplace=True)
    
    df = df.dropna(how='all')
    
    df_long = df.melt(
        id_vars=['mesorregiao'],
        var_name='variavel_ano_tipo',
        value_name='valor'
    )
    
    df_long[['variavel', 'ano', 'tipo']] = df_long['variavel_ano_tipo'].str.rsplit('_', n=2, expand=True)
    
    df_long = df_long.drop(columns=['variavel_ano_tipo'])
    df_long['valor'] = pd.to_numeric(df_long['valor'], errors='coerce')


    df_final = df_long.pivot_table(
        index=['mesorregiao', 'ano', 'tipo'],
        columns='variavel',
        values='valor'
    ).reset_index()
    
    return df_final

In [233]:
dfs_arabica = []

for arquivo in arquivos_arabica:
    
    try:
        df = trata_excel(arquivo)
        
        dfs_arabica.append(df)
        
    except Exception as e:
        print(f"Erro em {arquivo}: {e}")

dfs_canephora = []

for arquivo in arquivos_canephora:
    
    try:
        df = trata_excel(arquivo)
        
        dfs_canephora.append(df)
        
    except Exception as e:
        print(f"Erro em {arquivo}: {e}")

In [234]:
dfs_arabica[0]

variavel,mesorregiao,ano,tipo,Variável - Área colhida (Hectares)
0,Agreste Paraibano (PB),2024,Café (em grão) Arábica,2.0
1,Agreste Pernambucano (PE),2012,Café (em grão) Arábica,2733.0
2,Agreste Pernambucano (PE),2013,Café (em grão) Arábica,2615.0
3,Agreste Pernambucano (PE),2014,Café (em grão) Arábica,2549.0
4,Agreste Pernambucano (PE),2015,Café (em grão) Arábica,2265.0
...,...,...,...,...
819,Zona da Mata (MG),2020,Café (em grão) Arábica,202022.0
820,Zona da Mata (MG),2021,Café (em grão) Arábica,190782.0
821,Zona da Mata (MG),2022,Café (em grão) Arábica,202730.0
822,Zona da Mata (MG),2023,Café (em grão) Arábica,201832.0


In [235]:
dfs_canephora[0]

variavel,mesorregiao,ano,tipo,Variável - Área colhida (Hectares)
0,Baixo Amazonas (PA),2012,Café (em grão) Canephora,255.0
1,Baixo Amazonas (PA),2013,Café (em grão) Canephora,213.0
2,Baixo Amazonas (PA),2014,Café (em grão) Canephora,138.0
3,Baixo Amazonas (PA),2015,Café (em grão) Canephora,56.0
4,Baixo Amazonas (PA),2016,Café (em grão) Canephora,60.0
...,...,...,...,...
400,Zona da Mata (MG),2020,Café (em grão) Canephora,522.0
401,Zona da Mata (MG),2021,Café (em grão) Canephora,504.0
402,Zona da Mata (MG),2022,Café (em grão) Canephora,488.0
403,Zona da Mata (MG),2023,Café (em grão) Canephora,498.0


In [236]:
df_final_arabica = reduce(lambda left, right: left.merge(right, on=['mesorregiao', 'ano', 'tipo']), dfs_arabica)
df_final_canephora = reduce(lambda left, right: left.merge(right, on=['mesorregiao', 'ano', 'tipo']), dfs_canephora)

In [237]:
df_final = pd.concat([df_final_arabica, df_final_canephora], ignore_index=True)

In [238]:
df_final['Variável - Valor da produção (Mil Reais)'] = df_final['Variável - Valor da produção (Mil Reais)'] * 1000
df_final['Variável - Quantidade produzida (Toneladas)'] = df_final['Variável - Quantidade produzida (Toneladas)'] * 1000

In [239]:
df_final.rename(columns={'Variável - Valor da produção (Mil Reais)': 'Variável - Valor da produção (Reais)',
                         'Variável - Quantidade produzida (Toneladas)': 'Variável - Quantidade produzida (Quilogramas)'
                         }, inplace=True)

In [240]:
df_final

variavel,mesorregiao,ano,tipo,Variável - Área colhida (Hectares),Variável - Área destinada à colheita (Hectares),Variável - Quantidade produzida (Quilogramas),Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais)
0,Agreste Paraibano (PB),2024,Café (em grão) Arábica,2.0,2.0,2000.0,1000.0,36000.0
1,Agreste Pernambucano (PE),2012,Café (em grão) Arábica,2733.0,2733.0,1075000.0,393.0,3851000.0
2,Agreste Pernambucano (PE),2013,Café (em grão) Arábica,2615.0,2615.0,771000.0,295.0,2439000.0
3,Agreste Pernambucano (PE),2014,Café (em grão) Arábica,2549.0,2549.0,693000.0,272.0,1618000.0
4,Agreste Pernambucano (PE),2015,Café (em grão) Arábica,2265.0,2265.0,867000.0,383.0,2325000.0
...,...,...,...,...,...,...,...,...
1224,Zona da Mata (MG),2020,Café (em grão) Canephora,522.0,522.0,1233000.0,2362.0,6623000.0
1225,Zona da Mata (MG),2021,Café (em grão) Canephora,504.0,504.0,1069000.0,2121.0,8679000.0
1226,Zona da Mata (MG),2022,Café (em grão) Canephora,488.0,488.0,1340000.0,2746.0,15327000.0
1227,Zona da Mata (MG),2023,Café (em grão) Canephora,498.0,498.0,1181000.0,2371.0,10643000.0


In [241]:
mesorregioes = gpd.read_file("C:/Users/gabriel_lc/Desktop/saffraz-k-means/dados-brutos/centroides\BR_Mesorregioes_2022/BR_Mesorregioes_2022.shp")
mesorregioes['centroide'] = mesorregioes.geometry.centroid
mesorregioes['latitude'] = mesorregioes.centroide.y
mesorregioes['longitude'] = mesorregioes.centroide.x

mesorregioes['NM_MESO'] = mesorregioes['NM_MESO'] + ' (' + mesorregioes['SIGLA_UF'] + ')'

mesorregioes = mesorregioes[['NM_MESO', 'latitude', 'longitude']]

C:\Users\gabriel_lc\AppData\Local\Temp\ipykernel_31616\3701214272.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  mesorregioes['centroide'] = mesorregioes.geometry.centroid


In [242]:
mesorregioes.rename(columns={'NM_MESO': 'mesorregiao'}, inplace=True)
mesorregioes.head()

,mesorregiao,latitude,longitude
0,Madeira-Guaporé (RO),-10.302659,-64.055361
1,Leste Rondoniense (RO),-11.406309,-61.861723
2,Vale do Juruá (AC),-8.539777,-71.773867
3,Vale do Acre (AC),-9.941532,-69.066201
4,Norte Amazonense (AM),-0.615226,-65.567522


In [243]:
df_final = df_final.merge(mesorregioes, on='mesorregiao', how='left')

In [244]:
df_final.head()

,mesorregiao,ano,tipo,Variável - Área colhida (Hectares),Variável - Área destinada à colheita (Hectares),Variável - Quantidade produzida (Quilogramas),Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),latitude,longitude
0,Agreste Paraibano (PB),2024,Café (em grão) Arábica,2.0,2.0,2000.0,1000.0,36000.0,-7.029713,-35.819545
1,Agreste Pernambucano (PE),2012,Café (em grão) Arábica,2733.0,2733.0,1075000.0,393.0,3851000.0,-8.510505,-36.416511
2,Agreste Pernambucano (PE),2013,Café (em grão) Arábica,2615.0,2615.0,771000.0,295.0,2439000.0,-8.510505,-36.416511
3,Agreste Pernambucano (PE),2014,Café (em grão) Arábica,2549.0,2549.0,693000.0,272.0,1618000.0,-8.510505,-36.416511
4,Agreste Pernambucano (PE),2015,Café (em grão) Arábica,2265.0,2265.0,867000.0,383.0,2325000.0,-8.510505,-36.416511


In [248]:
def trata_temperatura(caminho):

    cabecalho = pd.read_csv(
        caminho,
        sep=';',
        encoding='latin1',
        nrows=8,
        header=None
    )


    latitude = cabecalho.iloc[4, 1]
    longitude = cabecalho.iloc[5, 1]
    altitude  = cabecalho.iloc[6, 1]


    df = pd.read_csv(
    caminho,
    sep=';',
    encoding='latin1',   # resolve os acentos bugados
    skiprows=8           # pula os metadados    
    )

    df['latitude'] = latitude
    df['longitude'] = longitude
    df['altitude'] = altitude

    nome_arquivo = os.path.basename(caminho)

    partes = nome_arquivo.split('_')

    regiao = partes[1]
    uf = partes[2]
    cidade = partes[4]
    ano = partes[5]

    df['regiao'] = regiao
    df['uf'] = uf
    df['cidade'] = cidade
    df['ano'] = ano

    if ano[-4:] in ['2019', '2020', '2021', '2022', '2023', '2024']:
        df['DATA (YYYY-MM-DD)'] = pd.to_datetime(df['Data']).dt.date

    df = df[['DATA (YYYY-MM-DD)', 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)', 'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)',
          'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)', 'UMIDADE RELATIVA DO AR, HORARIA (%)', 
          'regiao', 'uf', 'cidade', 'ano', 'latitude', 'longitude', 'altitude']]
    
    cols = [
        'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
        'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)',
        'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)',
        'altitude'
    ]

    for col in cols:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(',', '.'),
            errors='coerce'
        )

    df['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].astype(float)
    df['TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)'] = df['TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)'].astype(float)
    df['TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)'] = df['TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)'].astype(float)
    df['altitude'] = df['altitude'].astype(float)

    df['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].clip(lower=0)
    df = df[df['TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)'] >= -100]
    df = df[df['TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)'] >= -100]
    df = df[df['UMIDADE RELATIVA DO AR, HORARIA (%)'] >= 0]
    df = df[df['UMIDADE RELATIVA DO AR, HORARIA (%)'] <= 100]
    df = df[df['altitude'] >= 0]

    df_diario = df.groupby('DATA (YYYY-MM-DD)').agg({
        'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'sum',
        'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'max',
        'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'min',
        'UMIDADE RELATIVA DO AR, HORARIA (%)': 'mean',
        'altitude': 'mean'
    }).reset_index()


    df_diario['DATA (YYYY-MM-DD)'] = pd.to_datetime(df_diario['DATA (YYYY-MM-DD)'])
    mask = df_diario['DATA (YYYY-MM-DD)'].dt.month >= 10
    soma = df_diario.loc[mask, 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].sum()
    df_diario['precipitacao_floracao'] = soma

    df_final = df_diario.agg({
        'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'sum',
        'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'mean',
        'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'mean',
        'UMIDADE RELATIVA DO AR, HORARIA (%)': 'mean',
        'precipitacao_floracao': 'mean',
        'altitude': 'mean'
    }).to_frame().T


    df_final['regiao'] = df['regiao'].iloc[0]
    df_final['uf'] = df['uf'].iloc[0]
    df_final['cidade'] = df['cidade'].iloc[0]
    df_final['ano'] = df['ano'].iloc[0]
    df_final['latitude'] = df['latitude'].iloc[0]
    df_final['longitude'] = df['longitude'].iloc[0]

    df_final = df_final.rename(columns={
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'precipitacao_total (mm)',
    'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'temperatura_maxima (°C)',
    'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'temperatura_minima (°C)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)': 'umidade_relativa (%)'})

    df_final['ano'] = pd.to_datetime(df_final['ano'], format='%d-%m-%Y')
    df_final['ano'] = df_final['ano'].dt.year

    return df_final

In [246]:
df = pd.read_csv(
    r'dados-brutos/dados-climaticos\2012\2012\INMET_CO_DF_A001_BRASILIA_01-01-2012_A_31-12-2012.CSV',
    sep=';',
    encoding='latin1',   # resolve os acentos bugados
    skiprows=8           # pula os metadados    
    )

In [249]:
dfs = []

for root, dirs, files in os.walk('dados-brutos/dados-climaticos'):
    for file in files:
        if file.endswith('.CSV'):
            
            caminho = os.path.join(root, file)
            
            try:
                df = trata_temperatura(caminho)
                
                if df is not None:
                    dfs.append(df)
            
            except Exception as e:
                print(f"Erro em {caminho}: {e}")

df_temp = pd.concat(dfs, ignore_index=True)

Erro em dados-brutos/dados-climaticos\2012\2012\INMET_NE_RN_A302_ARQ.SAO PEDRO E SAO PAULO_01-01-2012_A_31-12-2012.CSV: single positional indexer is out-of-bounds
Erro em dados-brutos/dados-climaticos\2012\2012\INMET_N_PA_A234_TUCUMA_01-01-2012_A_31-12-2012.CSV: single positional indexer is out-of-bounds
Erro em dados-brutos/dados-climaticos\2013\INMET_NE_RN_A302_ARQ.SAO PEDRO E SAO PAULO_01-01-2013_A_31-12-2013.CSV: single positional indexer is out-of-bounds
Erro em dados-brutos/dados-climaticos\2013\INMET_N_PA_A234_TUCUMA_01-01-2013_A_31-12-2013.CSV: single positional indexer is out-of-bounds
Erro em dados-brutos/dados-climaticos\2013\INMET_SE_MG_F501_BELO HORIZONTE - CERCADINHO_27-12-2013_A_31-12-2013.CSV: single positional indexer is out-of-bounds
Erro em dados-brutos/dados-climaticos\2014\2014\INMET_NE_RN_A302_ARQ.SAO PEDRO E SAO PAULO_01-01-2014_A_31-12-2014.CSV: single positional indexer is out-of-bounds
Erro em dados-brutos/dados-climaticos\2014\2014\INMET_N_PA_A234_TUCUMA_01-0

In [250]:
df_temp.tail()

,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude,regiao,uf,cidade,ano,latitude,longitude
6745,286.6,22.911268,18.392254,91.048732,38.8,34.36,S,SC,Laguna - Farol de Santa Marta,2024,"-28,60444444","-48,81333333"
6746,1675.8,25.382240,16.835792,85.811543,147.8,2.00,S,SC,ARARANGUA,2024,"-28,931353","-49,49792"
6747,2104.4,25.283014,17.926027,82.882414,631.4,9.76,S,SC,ITAJAI,2024,"-26,95083333","-48,76194444"
6748,891.4,28.567910,19.239552,75.571464,0.0,679.00,S,SC,CHAPECO,2024,"-27,0853111","-52,6357111"
6749,1899.8,23.781967,13.808470,78.273432,503.2,963.00,S,SC,CAMPOS NOVOS,2024,"-27,3886111","-51,21583333"


In [251]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # raio da Terra em km
    
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    
    return R * c 

In [252]:
mesorregioes

,mesorregiao,latitude,longitude
0,Madeira-Guaporé (RO),-10.302659,-64.055361
1,Leste Rondoniense (RO),-11.406309,-61.861723
2,Vale do Juruá (AC),-8.539777,-71.773867
3,Vale do Acre (AC),-9.941532,-69.066201
4,Norte Amazonense (AM),-0.615226,-65.567522
...,...,...,...
134,Norte Goiano (GO),-13.895544,-48.347095
135,Centro Goiano (GO),-15.981377,-49.778494
136,Leste Goiano (GO),-15.279044,-47.512262
137,Sul Goiano (GO),-17.743173,-50.504003


In [253]:
cols = [
        'latitude',
        'longitude'
    ]

for col in cols:
    df_temp[col] = pd.to_numeric(
        df_temp[col].astype(str).str.replace(',', '.'),
        errors='coerce'
    )

df_temp['latitude'] = df_temp['latitude'].astype(float)
df_temp['longitude'] = df_temp['longitude'].astype(float)

In [254]:
df_temp_teste = df_temp[df_temp['ano'] == 2019]

In [255]:
df_temp_teste

,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude,regiao,uf,cidade,ano,latitude,longitude
3518,1369.4,28.050685,16.936438,64.033021,502.2,1160.96,CO,DF,BRASILIA,2019,-15.789343,-47.925756
3519,1466.2,28.323691,18.042424,61.651188,497.8,1143.00,CO,DF,BRAZLANDIA,2019,-15.599722,-48.131111
3520,1369.6,29.571154,15.793407,66.829497,630.6,1030.36,CO,DF,AGUAS EMENDADAS,2019,-15.596491,-47.625801
3521,1247.8,29.413425,16.670959,64.407435,456.2,990.00,CO,DF,GAMA (PONTE ALTA),2019,-15.935278,-48.137500
3522,1187.4,28.988493,17.292603,67.867600,543.2,1043.00,CO,DF,PARANOA (COOPA-DF),2019,-16.012222,-47.557417
...,...,...,...,...,...,...,...,...,...,...,...,...
4101,829.0,25.970674,17.606452,85.016268,102.0,2.00,S,SC,ARARANGUA,2019,-28.931353,-49.497920
4102,1587.6,25.850685,17.919452,83.017808,376.0,9.76,S,SC,ITAJAI,2019,-26.950924,-48.762031
4103,1296.8,22.207647,12.949118,92.295600,240.6,881.00,S,SC,RANCHO QUEIMADO,2019,-27.678507,-49.042027
4104,1409.4,25.073101,15.254747,72.083480,530.2,680.00,S,SC,CHAPECO,2019,-27.955278,-52.635556


In [256]:
def achar_microrregiao(lat_est, lon_est, df_centroides):
    menor_distancia = float('inf')
    microrregiao_mais_proxima = None
    
    for _, row in df_centroides.iterrows():
        dist = haversine(lat_est, lon_est, row['latitude'], row['longitude'])
        
        if dist < menor_distancia:
            menor_distancia = dist
            microrregiao_mais_proxima = row['mesorregiao']
    
    return microrregiao_mais_proxima, menor_distancia


# Aplica para todas as estações
df_temp_teste['mesorregiao'] = df_temp_teste.apply(
    lambda row: achar_microrregiao(row['latitude'], row['longitude'], mesorregioes)[0],
    axis=1
)

print(mesorregioes)

                mesorregiao   latitude  longitude
0      Madeira-Guaporé (RO) -10.302659 -64.055361
1    Leste Rondoniense (RO) -11.406309 -61.861723
2        Vale do Juruá (AC)  -8.539777 -71.773867
3         Vale do Acre (AC)  -9.941532 -69.066201
4     Norte Amazonense (AM)  -0.615226 -65.567522
..                      ...        ...        ...
134       Norte Goiano (GO) -13.895544 -48.347095
135      Centro Goiano (GO) -15.981377 -49.778494
136       Leste Goiano (GO) -15.279044 -47.512262
137         Sul Goiano (GO) -17.743173 -50.504003
138   Distrito Federal (DF) -15.781166 -47.796851

[139 rows x 3 columns]


C:\Users\gabriel_lc\AppData\Local\Temp\ipykernel_31616\3574635126.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp_teste['mesorregiao'] = df_temp_teste.apply(


In [257]:
df_temp_teste

,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude,regiao,uf,cidade,ano,latitude,longitude,mesorregiao
3518,1369.4,28.050685,16.936438,64.033021,502.2,1160.96,CO,DF,BRASILIA,2019,-15.789343,-47.925756,Distrito Federal (DF)
3519,1466.2,28.323691,18.042424,61.651188,497.8,1143.00,CO,DF,BRAZLANDIA,2019,-15.599722,-48.131111,Distrito Federal (DF)
3520,1369.6,29.571154,15.793407,66.829497,630.6,1030.36,CO,DF,AGUAS EMENDADAS,2019,-15.596491,-47.625801,Distrito Federal (DF)
3521,1247.8,29.413425,16.670959,64.407435,456.2,990.00,CO,DF,GAMA (PONTE ALTA),2019,-15.935278,-48.137500,Distrito Federal (DF)
3522,1187.4,28.988493,17.292603,67.867600,543.2,1043.00,CO,DF,PARANOA (COOPA-DF),2019,-16.012222,-47.557417,Distrito Federal (DF)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4101,829.0,25.970674,17.606452,85.016268,102.0,2.00,S,SC,ARARANGUA,2019,-28.931353,-49.497920,Sul Catarinense (SC)
4102,1587.6,25.850685,17.919452,83.017808,376.0,9.76,S,SC,ITAJAI,2019,-26.950924,-48.762031,Vale do Itajaí (SC)
4103,1296.8,22.207647,12.949118,92.295600,240.6,881.00,S,SC,RANCHO QUEIMADO,2019,-27.678507,-49.042027,Grande Florianópolis (SC)
4104,1409.4,25.073101,15.254747,72.083480,530.2,680.00,S,SC,CHAPECO,2019,-27.955278,-52.635556,Noroeste Rio-grandense (RS)


In [258]:
df_temp = df_temp.merge(
    df_temp_teste[['mesorregiao', 'regiao', 'uf', 'cidade']],
    on=['regiao', 'uf', 'cidade'],
    how='left'
)

In [259]:
df_temp

,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude,regiao,uf,cidade,ano,latitude,longitude,mesorregiao
0,1307.4,27.054645,16.604098,65.373758,587.8,1159.54,CO,DF,BRASILIA,2012,-15.789444,-47.925833,Distrito Federal (DF)
1,1284.0,28.527049,15.517213,67.855631,565.2,1200.00,CO,DF,AGUAS EMENDADAS,2012,-15.596389,-47.625833,Distrito Federal (DF)
2,1708.8,30.822678,17.804918,67.062033,585.8,770.00,CO,GO,GOIANIA,2012,-16.642778,-49.220000,Centro Goiano (GO)
3,685.0,29.829781,16.812842,68.223585,328.2,771.42,CO,GO,MORRINHOS,2012,-17.716667,-49.100000,Sul Goiano (GO)
4,505.4,31.284573,19.514050,62.458362,332.6,488.51,CO,GO,SAO SIMAO,2012,-18.966667,-50.616667,Sul Goiano (GO)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6745,286.6,22.911268,18.392254,91.048732,38.8,34.36,S,SC,Laguna - Farol de Santa Marta,2024,-28.604444,-48.813333,Sul Catarinense (SC)
6746,1675.8,25.382240,16.835792,85.811543,147.8,2.00,S,SC,ARARANGUA,2024,-28.931353,-49.497920,Sul Catarinense (SC)
6747,2104.4,25.283014,17.926027,82.882414,631.4,9.76,S,SC,ITAJAI,2024,-26.950833,-48.761944,Vale do Itajaí (SC)
6748,891.4,28.567910,19.239552,75.571464,0.0,679.00,S,SC,CHAPECO,2024,-27.085311,-52.635711,Noroeste Rio-grandense (RS)


In [260]:
df_temp['mesorregiao'].nunique()

139

In [261]:
mesorregioes['mesorregiao'].nunique()

139

In [262]:
df_temp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6750 entries, 0 to 6749
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   precipitacao_total (mm)  6750 non-null   float64
 1   temperatura_maxima (°C)  6750 non-null   float64
 2   temperatura_minima (°C)  6750 non-null   float64
 3   umidade_relativa (%)     6750 non-null   float64
 4   precipitacao_floracao    6750 non-null   float64
 5   altitude                 6750 non-null   float64
 6   regiao                   6750 non-null   object 
 7   uf                       6750 non-null   object 
 8   cidade                   6750 non-null   object 
 9   ano                      6750 non-null   int32  
 10  latitude                 6750 non-null   float64
 11  longitude                6750 non-null   float64
 12  mesorregiao              6607 non-null   object 
dtypes: float64(8), int32(1), object(4)
memory usage: 659.3+ KB


In [264]:
df_temp_agg = df_temp.groupby(['mesorregiao', 'ano']).agg({
    'precipitacao_total (mm)': 'mean',
    'temperatura_maxima (°C)': 'mean',
    'temperatura_minima (°C)': 'mean',
    'umidade_relativa (%)': 'mean',
    'precipitacao_floracao': 'mean',
    'altitude': 'mean'
}).reset_index()

In [265]:
df_temp_agg

,mesorregiao,ano,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude
0,Agreste Alagoano (AL),2012,601.533333,30.401879,21.205324,73.892771,35.266667,196.666667
1,Agreste Alagoano (AL),2013,1007.266667,30.712952,21.571794,76.156624,224.200000,196.666667
2,Agreste Alagoano (AL),2014,1036.800000,30.174338,21.319909,78.302930,196.066667,196.666667
3,Agreste Alagoano (AL),2015,574.600000,32.988236,23.222919,71.140840,38.133333,196.666667
4,Agreste Alagoano (AL),2016,538.066667,31.911789,22.327992,65.495402,30.733333,196.666667
...,...,...,...,...,...,...,...,...
1769,Zona da Mata (MG),2020,1660.733333,29.007442,18.022448,78.142019,505.666667,463.856667
1770,Zona da Mata (MG),2021,896.600000,29.200172,17.050388,77.010000,349.800000,463.856667
1771,Zona da Mata (MG),2022,1326.733333,29.056661,17.429160,76.393595,734.600000,463.856667
1772,Zona da Mata (MG),2023,1327.266667,29.921461,18.544932,77.370603,405.400000,463.856667


In [266]:
df_temp_agg[df_temp_agg['mesorregiao'].str.contains('(SC)')]

C:\Users\gabriel_lc\AppData\Local\Temp\ipykernel_31616\3514436738.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_temp_agg[df_temp_agg['mesorregiao'].str.contains('(SC)')]


,mesorregiao,ano,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude
433,Grande Florianópolis (SC),2012,1393.00,25.562842,18.126503,77.570397,284.80,1.800
434,Grande Florianópolis (SC),2013,1601.80,24.890137,17.440822,77.828063,262.80,1.800
435,Grande Florianópolis (SC),2014,1530.20,25.803836,18.530685,78.482802,388.60,1.800
436,Grande Florianópolis (SC),2015,2275.40,25.530959,18.756438,80.173554,692.40,1.800
437,Grande Florianópolis (SC),2016,1390.00,22.229930,13.749866,80.844010,585.90,441.400
...,...,...,...,...,...,...,...,...
1705,Vale do Itajaí (SC),2020,1034.40,24.668980,15.302514,82.191869,292.95,288.365
1706,Vale do Itajaí (SC),2021,760.05,23.235632,15.737916,85.148857,76.35,288.365
1707,Vale do Itajaí (SC),2022,1171.70,26.073810,16.485651,80.541036,577.70,288.365
1708,Vale do Itajaí (SC),2023,1843.35,25.063638,16.165525,84.111455,468.25,288.365


In [267]:
df_final.head()

,mesorregiao,ano,tipo,Variável - Área colhida (Hectares),Variável - Área destinada à colheita (Hectares),Variável - Quantidade produzida (Quilogramas),Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),latitude,longitude
0,Agreste Paraibano (PB),2024,Café (em grão) Arábica,2.0,2.0,2000.0,1000.0,36000.0,-7.029713,-35.819545
1,Agreste Pernambucano (PE),2012,Café (em grão) Arábica,2733.0,2733.0,1075000.0,393.0,3851000.0,-8.510505,-36.416511
2,Agreste Pernambucano (PE),2013,Café (em grão) Arábica,2615.0,2615.0,771000.0,295.0,2439000.0,-8.510505,-36.416511
3,Agreste Pernambucano (PE),2014,Café (em grão) Arábica,2549.0,2549.0,693000.0,272.0,1618000.0,-8.510505,-36.416511
4,Agreste Pernambucano (PE),2015,Café (em grão) Arábica,2265.0,2265.0,867000.0,383.0,2325000.0,-8.510505,-36.416511


In [268]:
df_temp_agg.head()

,mesorregiao,ano,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude
0,Agreste Alagoano (AL),2012,601.533333,30.401879,21.205324,73.892771,35.266667,196.666667
1,Agreste Alagoano (AL),2013,1007.266667,30.712952,21.571794,76.156624,224.200000,196.666667
2,Agreste Alagoano (AL),2014,1036.800000,30.174338,21.319909,78.302930,196.066667,196.666667
3,Agreste Alagoano (AL),2015,574.600000,32.988236,23.222919,71.140840,38.133333,196.666667
4,Agreste Alagoano (AL),2016,538.066667,31.911789,22.327992,65.495402,30.733333,196.666667


In [269]:
df_final['ano'] = df_final['ano'].astype(int)

df_final = df_final.merge(df_temp_agg, on=['mesorregiao', 'ano'], how='left')

In [270]:
df_final

,mesorregiao,ano,tipo,Variável - Área colhida (Hectares),Variável - Área destinada à colheita (Hectares),Variável - Quantidade produzida (Quilogramas),Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),latitude,longitude,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude
0,Agreste Paraibano (PB),2024,Café (em grão) Arábica,2.0,2.0,2000.0,1000.0,36000.0,-7.029713,-35.819545,768.800000,28.629075,20.254114,82.511918,54.100000,559.810000
1,Agreste Pernambucano (PE),2012,Café (em grão) Arábica,2733.0,2733.0,1075000.0,393.0,3851000.0,-8.510505,-36.416511,232.066667,29.143534,18.353461,70.905596,12.333333,684.486667
2,Agreste Pernambucano (PE),2013,Café (em grão) Arábica,2615.0,2615.0,771000.0,295.0,2439000.0,-8.510505,-36.416511,618.066667,29.036885,18.879336,73.331692,106.800000,684.486667
3,Agreste Pernambucano (PE),2014,Café (em grão) Arábica,2549.0,2549.0,693000.0,272.0,1618000.0,-8.510505,-36.416511,606.266667,28.628146,18.599447,74.939452,116.933333,684.486667
4,Agreste Pernambucano (PE),2015,Café (em grão) Arábica,2265.0,2265.0,867000.0,383.0,2325000.0,-8.510505,-36.416511,403.000000,29.760529,18.845805,70.811290,57.933333,684.486667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1224,Zona da Mata (MG),2020,Café (em grão) Canephora,522.0,522.0,1233000.0,2362.0,6623000.0,-21.010901,-42.819740,1660.733333,29.007442,18.022448,78.142019,505.666667,463.856667
1225,Zona da Mata (MG),2021,Café (em grão) Canephora,504.0,504.0,1069000.0,2121.0,8679000.0,-21.010901,-42.819740,896.600000,29.200172,17.050388,77.010000,349.800000,463.856667
1226,Zona da Mata (MG),2022,Café (em grão) Canephora,488.0,488.0,1340000.0,2746.0,15327000.0,-21.010901,-42.819740,1326.733333,29.056661,17.429160,76.393595,734.600000,463.856667
1227,Zona da Mata (MG),2023,Café (em grão) Canephora,498.0,498.0,1181000.0,2371.0,10643000.0,-21.010901,-42.819740,1327.266667,29.921461,18.544932,77.370603,405.400000,463.856667


In [271]:
df_final[df_final['altitude'].isna()]

,mesorregiao,ano,tipo,Variável - Área colhida (Hectares),Variável - Área destinada à colheita (Hectares),Variável - Quantidade produzida (Quilogramas),Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),latitude,longitude,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude
51,Assis (SP),2021,Café (em grão) Arábica,17415.0,17432.0,28009000.0,1608.0,2.917180e+08,-22.768414,-50.121347,NaN,NaN,NaN,NaN,NaN,NaN
52,Assis (SP),2022,Café (em grão) Arábica,17420.0,17437.0,27514000.0,1579.0,3.054490e+08,-22.768414,-50.121347,NaN,NaN,NaN,NaN,NaN,NaN
77,Campinas (SP),2021,Café (em grão) Arábica,53532.0,53648.0,85049000.0,1589.0,1.254622e+09,-22.220282,-46.989825,NaN,NaN,NaN,NaN,NaN,NaN
218,Centro-Sul Paranaense (PR),2012,Café (em grão) Arábica,1.0,1.0,1000.0,1000.0,5.000000e+03,-25.487813,-51.972895,NaN,NaN,NaN,NaN,NaN,NaN
477,Norte Fluminense (RJ),2012,Café (em grão) Arábica,105.0,105.0,151000.0,1438.0,4.400000e+05,-21.813031,-41.501991,NaN,NaN,NaN,NaN,NaN,NaN
478,Norte Fluminense (RJ),2013,Café (em grão) Arábica,89.0,89.0,105000.0,1180.0,3.430000e+05,-21.813031,-41.501991,NaN,NaN,NaN,NaN,NaN,NaN
479,Norte Fluminense (RJ),2014,Café (em grão) Arábica,63.0,93.0,97000.0,1540.0,2.740000e+05,-21.813031,-41.501991,NaN,NaN,NaN,NaN,NaN,NaN
480,Norte Fluminense (RJ),2015,Café (em grão) Arábica,63.0,63.0,97000.0,1540.0,2.740000e+05,-21.813031,-41.501991,NaN,NaN,NaN,NaN,NaN,NaN
481,Norte Fluminense (RJ),2016,Café (em grão) Arábica,63.0,63.0,97000.0,1540.0,4.090000e+05,-21.813031,-41.501991,NaN,NaN,NaN,NaN,NaN,NaN
482,Norte Fluminense (RJ),2017,Café (em grão) Arábica,65.0,65.0,103000.0,1585.0,7.200000e+05,-21.813031,-41.501991,NaN,NaN,NaN,NaN,NaN,NaN


In [272]:
df_final = df_final.sort_values(by=['mesorregiao', 'ano', 'tipo']).reset_index(drop=True)

df_final['precipitacao_total (mm)'] = df_final['precipitacao_total (mm)'].bfill()
df_final['temperatura_maxima (°C)'] = df_final['temperatura_maxima (°C)'].bfill()
df_final['temperatura_minima (°C)'] = df_final['temperatura_minima (°C)'].bfill()
df_final['umidade_relativa (%)'] = df_final['umidade_relativa (%)'].bfill()
df_final['altitude'] = df_final['altitude'].bfill()

In [273]:
df_final[df_final['altitude'].isna()]

,mesorregiao,ano,tipo,Variável - Área colhida (Hectares),Variável - Área destinada à colheita (Hectares),Variável - Quantidade produzida (Quilogramas),Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),latitude,longitude,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude


In [274]:
df_final[df_final['mesorregiao'].str.contains('Sudoeste Amazonense')]

,mesorregiao,ano,tipo,Variável - Área colhida (Hectares),Variável - Área destinada à colheita (Hectares),Variável - Quantidade produzida (Quilogramas),Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),latitude,longitude,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude
904,Sudoeste Amazonense (AM),2012,Café (em grão) Canephora,108.0,131.0,108000.0,1000.0,241000.0,-5.180785,-69.35668,693.200000,32.257724,22.260976,81.921297,608.8,143.00
905,Sudoeste Amazonense (AM),2013,Café (em grão) Canephora,70.0,108.0,56000.0,800.0,168000.0,-5.180785,-69.35668,2369.000000,31.049315,22.024384,80.183440,814.0,143.00
906,Sudoeste Amazonense (AM),2014,Café (em grão) Canephora,80.0,80.0,96000.0,1200.0,266000.0,-5.180785,-69.35668,2218.600000,30.990909,22.102204,70.406612,275.2,143.00
907,Sudoeste Amazonense (AM),2015,Café (em grão) Canephora,30.0,30.0,36000.0,1200.0,137000.0,-5.180785,-69.35668,2779.400000,31.900548,22.301644,65.677770,808.0,143.00
908,Sudoeste Amazonense (AM),2016,Café (em grão) Canephora,30.0,30.0,36000.0,1200.0,137000.0,-5.180785,-69.35668,786.400000,31.425743,23.499010,71.101247,0.0,143.00
909,Sudoeste Amazonense (AM),2017,Café (em grão) Canephora,30.0,30.0,36000.0,1200.0,162000.0,-5.180785,-69.35668,96.400000,31.576667,23.525000,72.051523,NaN,121.54
910,Sudoeste Amazonense (AM),2020,Café (em grão) Canephora,31.0,31.0,37000.0,1194.0,144000.0,-5.180785,-69.35668,96.400000,31.576667,23.525000,72.051523,0.0,121.54
911,Sudoeste Amazonense (AM),2021,Café (em grão) Arábica,10.0,21.0,13000.0,1300.0,128000.0,-5.180785,-69.35668,1171.733333,31.669913,19.965564,73.463972,NaN,377.25
912,Sudoeste Amazonense (AM),2022,Café (em grão) Arábica,4.0,4.0,8000.0,2000.0,67000.0,-5.180785,-69.35668,1171.733333,31.669913,19.965564,73.463972,NaN,377.25
913,Sudoeste Amazonense (AM),2022,Café (em grão) Canephora,12.0,19.0,15000.0,1250.0,79000.0,-5.180785,-69.35668,1171.733333,31.669913,19.965564,73.463972,NaN,377.25


In [275]:
df_final['aproveitamento_colheita'] = df_final['Variável - Área colhida (Hectares)'] / df_final['Variável - Área destinada à colheita (Hectares)']

df_final['amplitude_termica'] = df_final['temperatura_maxima (°C)'] - df_final['temperatura_minima (°C)']

df_final['preco_medio_kg'] = df_final['Variável - Valor da produção (Reais)'] / df_final['Variável - Quantidade produzida (Quilogramas)']

In [278]:
df_final[df_final['ano'] == 2024]

,mesorregiao,ano,tipo,Variável - Área colhida (Hectares),Variável - Área destinada à colheita (Hectares),Variável - Quantidade produzida (Quilogramas),Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),latitude,longitude,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude,aproveitamento_colheita,amplitude_termica,preco_medio_kg
0,Agreste Paraibano (PB),2024,Café (em grão) Arábica,2.0,2.0,2000.0,1000.0,3.600000e+04,-7.029713,-35.819545,768.800000,28.629075,20.254114,82.511918,54.100000,559.810000,1.0,8.374962,18.000000
13,Agreste Pernambucano (PE),2024,Café (em grão) Arábica,875.0,875.0,534000.0,610.0,1.299900e+07,-8.510505,-36.416511,265.733333,28.736046,19.267671,77.553712,22.800000,787.910000,1.0,9.468375,24.342697
15,Agreste Potiguar (RN),2024,Café (em grão) Arábica,3.0,3.0,7000.0,2333.0,1.050000e+05,-5.999192,-35.823165,23.400000,33.861000,23.648000,56.963968,22.600000,227.490000,1.0,10.213000,15.000000
28,Araraquara (SP),2024,Café (em grão) Arábica,2212.0,2212.0,3704000.0,1675.0,8.111400e+07,-21.792941,-48.309913,301.300000,31.063635,17.256135,66.191132,37.100000,606.370000,1.0,13.807499,21.899028
41,Araçatuba (SP),2024,Café (em grão) Arábica,640.0,640.0,557000.0,870.0,6.689000e+06,-21.066546,-50.813202,1058.200000,33.358470,19.729326,62.165858,546.733333,372.966667,1.0,13.629144,12.008977
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1176,Vale do Paraíba Paulista (SP),2024,Café (em grão) Arábica,3.0,3.0,5000.0,1667.0,5.800000e+04,-23.069587,-45.309119,1476.366667,26.238421,14.084896,74.496100,606.466667,1112.438333,1.0,12.153525,11.600000
1201,Vale do Rio Doce (MG),2024,Café (em grão) Arábica,61258.0,61258.0,83369000.0,1361.0,1.773457e+09,-18.985011,-42.047518,1308.700000,30.418904,19.658056,72.431584,602.000000,538.397500,1.0,10.760848,21.272379
1202,Vale do Rio Doce (MG),2024,Café (em grão) Canephora,9393.0,9393.0,21241000.0,2261.0,3.755110e+08,-18.985011,-42.047518,1308.700000,30.418904,19.658056,72.431584,602.000000,538.397500,1.0,10.760848,17.678593
1227,Zona da Mata (MG),2024,Café (em grão) Arábica,205375.0,205375.0,282595000.0,1376.0,5.693510e+09,-21.010901,-42.819740,1511.800000,30.457941,18.854546,76.623899,654.066667,463.856667,1.0,11.603395,20.147243
